Import packages and open data file

In [52]:
import pandas as pd              # Importing and organizing data
import numpy as np               # Numerical calculations
import matplotlib.pyplot as plt  # Creating graphs
import seaborn as sns            # Statistical graphs, built on Matplotlib

In [53]:
picudata = pd.read_csv('Deidentified PICU AC pulled 16 nov 26.csv')

Data Cleaning

In [54]:
#1 converting csv time text to datetime
timeColumns = ['PAT_RECEIVED', 'TRANS_ORDER_START', 'BED_ASSIGNED']
for column in timeColumns:
    picudata[column] = pd.to_datetime(picudata[column],format='%m/%d/%y %H:%M',errors='coerce')
#print(picudata[timeColumns].dtypes)  # test

In [55]:
#2 removing time columns missing data
print("Number of rows pre missing data clean:", len(picudata))
picudata = picudata.dropna(subset=timeColumns)
print("Number of rows post missing data clean:", len(picudata))

Number of rows pre missing data clean: 9819
Number of rows post missing data clean: 9477


In [56]:
#3 removing trailing spaces from unit and department names
textColumns = ['PICU_UNIT', 'TRANS_DEPT', 'TRANS_SERV', 'to_unit_descr']
for column in textColumns:
    picudata[column] = picudata[column].str.strip()

In [57]:
#4 creating calculated transfer time columns
# Transfer order to bed assignment
picudata['OrderToBedMin'] = picudata['BED_ASSIGNED'] - picudata['TRANS_ORDER_START']
picudata['OrderToBedMin'] = picudata['OrderToBedMin'].dt.total_seconds() / 60

# Transfer order to patient received
picudata['OrderToReceivedMin'] = picudata['PAT_RECEIVED'] - picudata['TRANS_ORDER_START']
picudata['OrderToReceivedMin'] = picudata['OrderToReceivedMin'].dt.total_seconds() / 60

# Bed assignment to patient received
picudata['BedToReceivedMin'] = picudata['PAT_RECEIVED'] - picudata['BED_ASSIGNED']
picudata['BedToReceivedMin'] = picudata['BedToReceivedMin'].dt.total_seconds() / 60

In [58]:
#5 calculate transfer duration and keep only rows <24hr
print("Number of rows pre transfer <=24hr clean:", len(picudata))
picudata = picudata[picudata['OrderToReceivedMin'] <= 1440]
print("Number of rows post transfer <=24hr clean:", len(picudata))

Number of rows pre transfer <=24hr clean: 9477
Number of rows post transfer <=24hr clean: 9469


In [59]:
#6 eliminate records where bed was assigned BEFORE order was placed
print("Number of rows pre bed before order clean:", len(picudata))
picudata = picudata[picudata['OrderToBedMin'] >= 0]
print("Number of rows post bed before order clean:", len(picudata))

Number of rows pre bed before order clean: 9469
Number of rows post bed before order clean: 9203
